In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [28]:
class InterpretableEnsemble:
    """
    Wraps Naive Bayes + LSTM ensemble with multi-layer attribution methods.
    Handles both TF-IDF text features AND numeric features.
    """

    def __init__(self, nb_model, lstm_model, vectorizer, tokenizer, training_df=None):
        self.nb = nb_model
        self.lstm = lstm_model
        self.vectorizer = vectorizer
        self.tokenizer = tokenizer
        self.training_df = training_df

        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

        print("✓ InterpretableEnsemble initialized")

    def predict_with_explanation(self, text,
                                 telecommuting=0,
                                 has_company_logo=0,
                                 has_questions=0,
                                 ensemble_strategy='rule_boosted'):  # NEW parameter
        """
        Comprehensive prediction with explanations.

        Args:
            text (str): Job posting text
            telecommuting (int): 0 or 1
            has_company_logo (int): 0 or 1
            has_questions (int): 0 or 1
            ensemble_strategy (str): 'simple', 'weighted', or 'rule_boosted' (default)

        Returns:
            dict: Fraud probability, decision, and multi-level explanations
        """

        # ===== PREPROCESS TEXT =====
        text_processed = self._preprocess_text(text)

        # ===== CALCULATE NUMERIC FEATURES =====
        location_fraud_ratio = 0.05
        character_count = len(text)

        # ===== GET NB PREDICTIONS =====
        nb_text_features = self.vectorizer.transform([text_processed])

        numeric_features = np.array([
            telecommuting,
            has_company_logo,
            has_questions,
            location_fraud_ratio,
            character_count
        ]).reshape(1, -1)

        from scipy.sparse import csr_matrix, hstack
        numeric_features_sparse = csr_matrix(numeric_features)
        nb_combined_features = hstack([nb_text_features, numeric_features_sparse])

        nb_prob = self.nb.predict_proba(nb_combined_features)[0, 1]

        # ===== GET LSTM PREDICTIONS =====
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        lstm_sequence = self.tokenizer.texts_to_sequences([text_processed])
        lstm_padded = pad_sequences(lstm_sequence, maxlen=200)
        lstm_prob = float(self.lstm.predict(lstm_padded, verbose=0)[0, 0])

        # ===== DETECT PATTERNS (needed for rule-boosting) =====
        patterns = self._detect_patterns(text)

        # ===== APPLY ENSEMBLE STRATEGY =====
        numeric_features_dict = {
            'telecommuting': telecommuting,
            'has_company_logo': has_company_logo,
            'has_questions': has_questions,
            'location_fraud_ratio': location_fraud_ratio,
            'character_count': character_count
        }

        if ensemble_strategy == 'simple':
            # Original: Simple average
            fraud_prob = (nb_prob + lstm_prob) / 2.0
            strategy_info = "Simple averaging (50/50)"
            boost_details = []

        elif ensemble_strategy == 'weighted':
            # Strategy 1: Fixed weights (LSTM=65%, NB=35%)
            fraud_prob = 0.35 * nb_prob + 0.65 * lstm_prob
            strategy_info = "Weighted averaging (NB=35%, LSTM=65%)"
            boost_details = []

        elif ensemble_strategy == 'rule_boosted':
            # Strategy 2: Rule-based boosting (RECOMMENDED)
            fraud_prob, strategy_info, boost_details = self._rule_boosted_ensemble(
                nb_prob, lstm_prob, numeric_features_dict, patterns, text
            )

        # ===== RISK CLASSIFICATION =====
        risk_level, risk_icon, recommendation = self._get_risk_classification(fraud_prob)

        # ===== GENERATE EXPLANATIONS =====
        explanation = self._generate_explanation(
            text=text,
            text_processed=text_processed,
            nb_features=nb_text_features,
            fraud_prob=fraud_prob,
            numeric_features=numeric_features_dict
        )

        # ===== RETURN ENHANCED RESULTS =====
        return {
            'fraud_probability': fraud_prob,
            'fraud_probability_pct': f"{fraud_prob*100:.1f}%",
            'risk_level': risk_level,           # NEW
            'risk_icon': risk_icon,             # NEW
            'recommendation': recommendation,    # NEW
            'ensemble_decision': 'FRAUD' if fraud_prob > 0.5 else 'REAL',
            'ensemble_strategy': strategy_info, # NEW
            'boost_details': boost_details,     # NEW
            'confidence': self._calculate_confidence(nb_prob, lstm_prob, fraud_prob),  # UPDATED
            'individual_predictions': {
                'naive_bayes_score': f"{nb_prob:.4f}",
                'lstm_score': f"{lstm_prob:.4f}",
                'agreement': abs(nb_prob - lstm_prob) < 0.15,
                'disagreement_margin': f"{abs(nb_prob - lstm_prob):.4f}"
            },
            'explanation': explanation,
            'features_used': numeric_features_dict
        }

    def _preprocess_text(self, text):
        """Preprocess text matching training pipeline"""
        tokens = word_tokenize(str(text).lower())
        tokens = [token for token in tokens
                 if token.isalpha() and token not in self.stop_words]
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens]
        return ' '.join(tokens)

    def _generate_explanation(self, text, text_processed, nb_features, fraud_prob, numeric_features):
        """Multi-level explanation generation"""
        return {
            'top_fraud_indicators': self._extract_top_features(nb_features),
            'sentence_attributions': self._highlight_risky_sentences(
                text, text_processed, nb_features
            ),
            'linguistic_flags': self._detect_patterns(text),
            'numeric_features_impact': self._explain_numeric_features(numeric_features),
            'confidence_reasoning': self._explain_confidence(fraud_prob)
        }

    def _extract_top_features(self, nb_features, top_n=10):
        """Extract top TF-IDF features driving fraud prediction"""
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        feature_log_prob_fraud = self.nb.feature_log_prob_[1][:5000]  # Only TF-IDF features
        feature_log_prob_real = self.nb.feature_log_prob_[0][:5000]

        tfidf_scores = nb_features.toarray()[0]
        fraud_weight = feature_log_prob_fraud - feature_log_prob_real
        contribution = tfidf_scores * fraud_weight

        top_indices = np.argsort(contribution)[-top_n:][::-1]

        top_features = {}
        for idx in top_indices:
            if contribution[idx] > 0:
                feature = feature_names[idx]
                top_features[feature] = {
                    'fraud_weight': float(fraud_weight[idx]),
                    'tfidf_score': float(tfidf_scores[idx]),
                    'contribution': float(contribution[idx])
                }

        return top_features

    def _highlight_risky_sentences(self, text, text_processed, nb_features, top_n=5):
        """Sentence-level attribution"""
        sentences = sent_tokenize(text)
        sentence_scores = []

        feature_names = np.array(self.vectorizer.get_feature_names_out())
        feature_log_prob_fraud = self.nb.feature_log_prob_[1][:5000]
        feature_log_prob_real = self.nb.feature_log_prob_[0][:5000]
        fraud_weights = feature_log_prob_fraud - feature_log_prob_real

        for sentence in sentences:
            sent_processed = self._preprocess_text(sentence)
            sent_features = self.vectorizer.transform([sent_processed])

            tfidf_scores = sent_features.toarray()[0]
            sent_fraud_score = np.sum(tfidf_scores * fraud_weights)
            sent_fraud_score = sent_fraud_score / (len(sent_processed.split()) + 1)

            patterns = self._detect_patterns_in_sentence(sentence)
            pattern_risk = len(patterns) * 0.15

            total_risk = max(0, min(1, sent_fraud_score + pattern_risk))

            sentence_scores.append({
                'sentence': sentence.strip(),
                'fraud_risk': total_risk,
                'fraud_risk_pct': f"{total_risk*100:.0f}%",
                'contributing_features': self._get_sentence_features(sent_features),
                'flagged_patterns': patterns
            })

        sentence_scores.sort(key=lambda x: x['fraud_risk'], reverse=True)
        return sentence_scores[:top_n]

    def _get_sentence_features(self, sent_features, top_n=3):
        """Extract top TF-IDF features from sentence"""
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        tfidf_scores = sent_features.toarray()[0]

        top_indices = np.argsort(tfidf_scores)[-top_n:][::-1]

        features = {}
        for idx in top_indices:
            if tfidf_scores[idx] > 0:
                features[feature_names[idx]] = float(tfidf_scores[idx])

        return features

    def _detect_patterns_in_sentence(self, sentence):
        """Detect linguistic red flags in sentence"""
        patterns = []
        sent_lower = sentence.lower()

        urgency_keywords = ['urgent', 'immediately', 'asap', 'deadline', 'quick', 'hurry']
        if any(keyword in sent_lower for keyword in urgency_keywords):
            patterns.append('Urgency indicator')

        payment_keywords = ['payment', 'fee', 'upfront', 'investment', 'wire', 'bitcoin']
        if any(keyword in sent_lower for keyword in payment_keywords):
            patterns.append('Payment request')

        vague_keywords = ['flexible', 'negotiable', 'not specified', 'tbd']
        if any(keyword in sent_lower for keyword in vague_keywords):
            patterns.append('Vague language')

        if '  ' in sentence or '..' in sentence or '!!' in sentence:
            patterns.append('Grammar issues')

        return patterns

    def _detect_patterns(self, text):
        """High-level pattern detection"""
        text_lower = text.lower()
        text_len = len(text)
        avg_sentence_len = np.mean([len(s.split()) for s in sent_tokenize(text)])

        return {
            'urgency_indicators': {
                'present': any(kw in text_lower for kw in ['urgent', 'asap', 'immediately']),
                'examples': [kw for kw in ['urgent', 'asap', 'immediately'] if kw in text_lower]
            },
            'payment_requests': {
                'present': any(kw in text_lower for kw in ['payment', 'fee', 'upfront']),
                'examples': [kw for kw in ['payment', 'fee', 'upfront'] if kw in text_lower]
            },
            'vague_language': {
                'present': any(kw in text_lower for kw in ['flexible', 'tbd', 'negotiable']),
                'examples': [kw for kw in ['flexible', 'tbd', 'negotiable'] if kw in text_lower]
            },
            'text_quality': {
                'word_count': len(text.split()),
                'character_count': text_len,
                'avg_sentence_length': round(avg_sentence_len, 2)
            }
        }

    def _explain_numeric_features(self, numeric_features):
        """Explain the impact of numeric features"""
        explanations = []

        if numeric_features['telecommuting'] == 1:
            explanations.append("✅ Job offers telecommuting (slightly lower fraud risk)")
        else:
            explanations.append("⚠️ No telecommuting mentioned")

        if numeric_features['has_company_logo'] == 1:
            explanations.append("✅ Job posting has company logo (lower fraud risk)")
        else:
            explanations.append("🚨 No company logo (common in fraud postings)")

        if numeric_features['has_questions'] == 1:
            explanations.append("✅ Job has screening questions (lower fraud risk)")
        else:
            explanations.append("⚠️ No screening questions")

        if numeric_features['character_count'] < 500:
            explanations.append("🚨 Very short job description (red flag)")
        elif numeric_features['character_count'] > 5000:
            explanations.append("⚠️ Unusually long description")
        else:
            explanations.append("✅ Normal description length")

        return {
            'features': numeric_features,
            'interpretations': explanations
        }

    def _explain_confidence(self, fraud_prob):
        """Provide reasoning for confidence level"""
        if fraud_prob > 0.8:
            confidence = 'VERY HIGH'
            reasoning = 'Strong fraud indicators detected'
            recommendation = 'DO NOT APPLY - High fraud risk'
        elif fraud_prob > 0.6:
            confidence = 'HIGH'
            reasoning = 'Multiple fraud indicators present'
            recommendation = 'PROCEED WITH CAUTION'
        elif fraud_prob > 0.4:
            confidence = 'MODERATE'
            reasoning = 'Mixed signals detected'
            recommendation = 'VERIFY company information'
        elif fraud_prob > 0.2:
            confidence = 'LOW'
            reasoning = 'Mostly safe indicators'
            recommendation = 'LIKELY SAFE'
        else:
            confidence = 'VERY LOW'
            reasoning = 'Strong legitimate indicators'
            recommendation = 'SAFE'

        return {
            'confidence_level': confidence,
            'fraud_probability': fraud_prob,
            'reasoning': reasoning,
            'recommendation': recommendation
        }

    def _rule_boosted_ensemble(self, nb_prob, lstm_prob, numeric_features, patterns, text):
        """
        Rule-based ensemble boosting with fraud heuristics.

        Adds fraud probability based on red flags:
        - Missing company features
        - Linguistic patterns (urgency, payment, vague language)
        - Grammar issues
        - Model disagreement indicators

        Returns:
            tuple: (fraud_prob, strategy_description, boost_details_list)
        """

        # Start with weighted average (favor LSTM slightly)
        base_prob = 0.25 * nb_prob + 0.75 * lstm_prob

        # Initialize boost tracking
        boost = 0.0
        boost_details = []

        # ===== RED FLAG CHECKS =====

        # Red Flag 1: Missing company logo (5% boost)
        if numeric_features['has_company_logo'] == 0:
            boost += 0.05
            boost_details.append("🚩 No company logo (+5%)")

        # Red Flag 2: No screening questions (5% boost)
        if numeric_features['has_questions'] == 0:
            boost += 0.05
            boost_details.append("🚩 No screening questions (+5%)")

        # Red Flag 3: Urgency indicators (8% boost)
        if patterns['urgency_indicators']['present']:
            keywords = ', '.join(patterns['urgency_indicators']['examples'][:3])
            boost += 0.08
            boost_details.append(f"🚩 Urgency language: {keywords} (+8%)")

        # Red Flag 4: Payment requests (12% boost - STRONG signal!)
        if patterns['payment_requests']['present']:
            keywords = ', '.join(patterns['payment_requests']['examples'][:3])
            boost += 0.12
            boost_details.append(f"🚩 Payment mentions: {keywords} (+12%)")

        # Red Flag 5: Vague language (4% boost)
        if patterns['vague_language']['present']:
            keywords = ', '.join(patterns['vague_language']['examples'][:2])
            boost += 0.04
            boost_details.append(f"🚩 Vague terms: {keywords} (+4%)")

        # Red Flag 6: Very short text (6% boost)
        if numeric_features['character_count'] < 300:
            boost += 0.06
            boost_details.append(f"🚩 Very short description ({numeric_features['character_count']} chars) (+6%)")

        # Red Flag 7: Grammar issues (5% boost)
        # Assuming _count_grammar_issues exists or will be added
        # For now, let's just make it a placeholder if it's not crucial to this fix
        # if self._count_grammar_issues(text) > 3:
        #    boost += 0.05
        #    boost_details.append(f"🚩 Grammar issues (+5%)")

        # Red Flag 8: Model disagreement with LSTM suspicious (10% boost)
        # This catches subtle frauds where LSTM sees patterns but NB doesn't
        if lstm_prob > 0.35 and nb_prob < 0.05:
            boost += 0.10
            boost_details.append(f"🚩 LSTM pattern detection (LSTM={lstm_prob:.1%}, NB={nb_prob:.1%}) (+10%)")

        # Apply boost (cap at 0.95 to avoid overconfidence)
        fraud_prob = min(base_prob + boost, 0.95)

        # Create strategy description
        if boost > 0:
            strategy_desc = f"Rule-boosted ensemble (base: {base_prob:.1%} + boost: {boost:.1%} = {fraud_prob:.1%})"
        else:
            strategy_desc = f"Weighted ensemble (no red flags detected)"

        return fraud_prob, strategy_desc, boost_details

    def _get_risk_classification(self, fraud_prob):
        """
        Three-tier risk classification system.

        Args:
            fraud_prob (float): Ensemble fraud probability (0-1)

        Returns:
            tuple: (risk_level, risk_icon, recommendation)
        """

        if fraud_prob > 0.60:
            return (
                'HIGH RISK',
                '🔴',
                'DO NOT APPLY - Strong fraud indicators detected'
            )
        elif fraud_prob > 0.20:
            return (
                'MEDIUM RISK',
                '🟡',
                'INVESTIGATE CAREFULLY - Verify company details, check reviews, research employer'
            )
        else:
            return (
                'LOW RISK',
                '🟢',
                'APPEARS SAFE - Standard application precautions recommended'
            )

    def _calculate_confidence(self, nb_prob, lstm_prob, fraud_prob):
        """
        Calculate confidence level based on model agreement and probability extremes.

        Args:
            nb_prob (float): Naive Bayes probability
            lstm_prob (float): LSTM probability
            fraud_prob (float): Final ensemble probability

        Returns:
            str: 'HIGH', 'MEDIUM', or 'LOW'
        """

        disagreement = abs(nb_prob - lstm_prob)

        # High confidence conditions:
        # 1. Models strongly agree (disagreement < 15%)
        # 2. Extreme probability (>80% or <10%)
        if disagreement < 0.15:
            return 'HIGH'
        elif fraud_prob > 0.80 or fraud_prob < 0.10:
            return 'HIGH'
        elif disagreement < 0.30:
            return 'MEDIUM'
        else:
            return 'LOW'

    # Helper method for rule_boosted_ensemble, assuming it exists or needs to be added.
    # Adding a dummy implementation if not present to avoid NameError for now.
    def _count_grammar_issues(self, text):
        # This is a placeholder. Implement actual grammar check logic if needed.
        # For simplicity, returning a fixed value or basic check for now.
        issues = 0
        if '  ' in text: issues += 1
        if '..' in text: issues += 1
        if '!!' in text: issues += 1
        return issues

print("✓ Updated InterpretableEnsemble class defined")

✓ Updated InterpretableEnsemble class defined


In [4]:
from google.colab import files
uploaded = files.upload()

Saving lstm_model.h5 to lstm_model.h5
Saving vectorizer.pkl to vectorizer.pkl
Saving tokenizer.pkl to tokenizer.pkl
Saving naive_bayes_model.pkl to naive_bayes_model.pkl


In [17]:
# Load models (use your saved model paths)
nb_model = pickle.load(open('naive_bayes_model.pkl', 'rb'))
lstm_model = load_model('lstm_model.h5')
vectorizer = pickle.load(open('vectorizer.pkl', 'rb'))
tokenizer = pickle.load(open('tokenizer.pkl', 'rb'))

print("✓ Models loaded")


✓ Models loaded


In [18]:
ensemble = InterpretableEnsemble(
    nb_model=nb_model,
    lstm_model=lstm_model,
    vectorizer=vectorizer,
    tokenizer=tokenizer
)


✓ InterpretableEnsemble initialized


Test Job (Synthetic)

In [19]:
# Test on fraudulent example
fraud_text = """
URGENT! WORK FROM HOME! Make $5000/week!

We are hiring immediately for remote customer service representatives.
No experience needed! Payment of $250 required upfront for training materials.

Contact us ASAP at temp@freemail.com
"""

# Enhanced test with full feature display
result = ensemble.predict_with_explanation(
    text=fraud_text,
    telecommuting=1,
    has_company_logo=0,
    has_questions=0,
    ensemble_strategy='rule_boosted'  # or 'simple' or 'weighted'
)

print("="*80)
print("FRAUD JOB POSTING DETECTION - INTERPRETABLE RESULTS")
print("="*80 + "\n")

print(f" FRAUD PROBABILITY: {result['fraud_probability_pct']}")
print(f" DECISION: {result['ensemble_decision']}")
print(f" CONFIDENCE: {result['confidence']}\n")

print(f" Model Agreement: {result['individual_predictions']['agreement']}")
print(f"   - Naive Bayes: {result['individual_predictions']['naive_bayes_score']}")
print(f"   - LSTM: {result['individual_predictions']['lstm_score']}")
print(f"   - Disagreement: {result['individual_predictions']['disagreement_margin']}\n")

print(f" Features Used:")
for feature, value in result['features_used'].items():
    print(f"   - {feature}: {value}")
print()

print(f" Top Fraud Indicators (TF-IDF Features):")
for i, (feature, details) in enumerate(list(result['explanation']['top_fraud_indicators'].items())[:7], 1):
    print(f"   {i}. '{feature}'")
    print(f"       Fraud weight: {details['fraud_weight']:.3f}")
    print(f"       TF-IDF score: {details['tfidf_score']:.3f}")
print()

print(f" Numeric Features Impact:")
for interp in result['explanation']['numeric_features_impact']['interpretations']:
    print(f"   {interp}")
print()

print(f" Linguistic Patterns Detected:")
patterns = result['explanation']['linguistic_flags']
if patterns['urgency_indicators']['present']:
    print(f"    Urgency indicators: {', '.join(patterns['urgency_indicators']['examples'])}")
if patterns['payment_requests']['present']:
    print(f"    Payment requests: {', '.join(patterns['payment_requests']['examples'])}")
if patterns['vague_language']['present']:
    print(f"    Vague language: {', '.join(patterns['vague_language']['examples'])}")
print(f"    Text quality: {patterns['text_quality']['word_count']} words, "
      f"{patterns['text_quality']['character_count']} characters")
print()

print(f" Risky Sentences (Sentence-Level Attribution):")
for i, sent in enumerate(result['explanation']['sentence_attributions'][:5], 1):
    print(f"\n   {i}. [{sent['fraud_risk_pct']}] {sent['sentence'][:100]}")
    if sent['flagged_patterns']:
        print(f"       Patterns: {', '.join(sent['flagged_patterns'])}")
    if sent['contributing_features']:
        print(f"       Key features: {', '.join(list(sent['contributing_features'].keys())[:3])}")
print()

print(f" Final Recommendation:")
reasoning = result['explanation']['confidence_reasoning']
print(f"   Level: {reasoning['confidence_level']}")
print(f"   Reasoning: {reasoning['reasoning']}")
print(f"    {reasoning['recommendation']}")

print("\n" + "="*80)

FRAUD JOB POSTING DETECTION - INTERPRETABLE RESULTS

 FRAUD PROBABILITY: 95.0%
 DECISION: FRAUD
 CONFIDENCE: HIGH

 Model Agreement: True
   - Naive Bayes: 0.8503
   - LSTM: 0.7722
   - Disagreement: 0.0781

 Features Used:
   - telecommuting: 1
   - has_company_logo: 0
   - has_questions: 0
   - location_fraud_ratio: 0.05
   - character_count: 231

 Top Fraud Indicators (TF-IDF Features):
   1. 'work home'
       Fraud weight: 3.225
       TF-IDF score: 0.324
   2. 'urgent'
       Fraud weight: 2.197
       TF-IDF score: 0.324
   3. 'service representative'
       Fraud weight: 1.615
       TF-IDF score: 0.323
   4. 'needed'
       Fraud weight: 1.513
       TF-IDF score: 0.220
   5. 'immediately'
       Fraud weight: 0.989
       TF-IDF score: 0.299
   6. 'home'
       Fraud weight: 1.372
       TF-IDF score: 0.212
   7. 'hiring'
       Fraud weight: 1.044
       TF-IDF score: 0.229

 Numeric Features Impact:
   ✅ Job offers telecommuting (slightly lower fraud ri

Test Job (Real, Fraud)

In [20]:
def combine_text_fields(row):
    """Combine all text fields into single string"""
    fields = [
        'title', 'location', 'company_profile', 'description',
        'requirements', 'benefits', 'required_experience',
        'required_education', 'industry', 'function'
    ]
    text_parts = []
    for field in fields:
        if pd.notna(row[field]):
            text_parts.append(str(row[field]))
    return ' '.join(text_parts) if text_parts else "unknown job"

def safe_display_field(row, field_name, max_length=100, default='N/A'):
    """Safely display a field that might be NaN"""
    value = row.get(field_name, default)
    if pd.isna(value):
        return default
    value_str = str(value)
    if len(value_str) > max_length:
        return value_str[:max_length] + '...'
    return value_str


In [9]:
import kagglehub
import pandas as pd

# Load the dataset (if not already loaded)
path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
df = pd.read_csv(f"{path}/fake_job_postings.csv")


100%|██████████| 16.1M/16.1M [00:00<00:00, 42.8MB/s]

Extracting files...


In [21]:
# Get a fraudulent job
fraud_jobs = df[df['fraudulent'] == 1]
fraud_sample = fraud_jobs.sample(1, random_state=42).iloc[0]

print("="*80)
print("TEST 1: FRAUDULENT JOB FROM DATASET")
print("="*80 + "\n")

print(f"Job ID: {fraud_sample['job_id']}")
print(f"Title: {fraud_sample['title']}")

# ✅ FIXED: Handle NaN values properly
company = fraud_sample.get('company_profile', 'N/A')
if pd.isna(company):
    company = 'N/A'
else:
    company = str(company)[:100] + '...'
print(f"Company: {company}")

print(f"Actual Label: FRAUDULENT\n")

# Use the combined text from dataset
fraud_text_real = combine_text_fields(fraud_sample)

# Predict on fraud_text_real
fraud_result = ensemble.predict_with_explanation(
    text=fraud_text_real,
    telecommuting=int(fraud_sample.get('telecommuting', 0)),
    has_company_logo=int(fraud_sample.get('has_company_logo', 0)),
    has_questions=int(fraud_sample.get('has_questions', 0)),
    ensemble_strategy='rule_boosted'
)

print(f"🔍 FRAUD PROBABILITY: {fraud_result['fraud_probability_pct']}")
print(f"{fraud_result['risk_icon']} RISK LEVEL: {fraud_result['risk_level']}")
print(f"📊 PREDICTED: {fraud_result['ensemble_decision']}")
print(f"✅ CONFIDENCE: {fraud_result['confidence']}")
print(f"🎯 CORRECT: {fraud_result['ensemble_decision'] == 'FRAUD'}\n")

# Show boost details if rule-boosted
if fraud_result.get('boost_details'):
    print(f"🚨 Red Flags Detected:")
    for detail in fraud_result['boost_details']:
        print(f"   {detail}")
    print()

print(f"🤖 Model Scores:")
print(f"   - Naive Bayes: {fraud_result['individual_predictions']['naive_bayes_score']}")
print(f"   - LSTM: {fraud_result['individual_predictions']['lstm_score']}")
print(f"   - Agreement: {fraud_result['individual_predictions']['agreement']}\n")

print(f"🎯 Strategy: {fraud_result.get('ensemble_strategy', 'N/A')}\n")

print(f"🚨 Top Fraud Indicators:")
for i, (feature, details) in enumerate(list(fraud_result['explanation']['top_fraud_indicators'].items())[:5], 1):
    print(f"   {i}. '{feature}': weight={details['fraud_weight']:.3f}")

print(f"\n⚠️ Risky Sentences:")
for sent in fraud_result['explanation']['sentence_attributions'][:3]:
    print(f"   [{sent['fraud_risk_pct']}] {sent['sentence'][:80]}...")
    if sent['flagged_patterns']:
        print(f"       🚩 {', '.join(sent['flagged_patterns'])}")

print(f"\n💡 Recommendation: {fraud_result['recommendation']}")


TEST 1: FRAUDULENT JOB FROM DATASET

Job ID: 17790
Title: customer service rep
Company: N/A
Actual Label: FRAUDULENT

🔍 FRAUD PROBABILITY: 95.0%
🔴 RISK LEVEL: HIGH RISK
📊 PREDICTED: FRAUD
✅ CONFIDENCE: HIGH
🎯 CORRECT: True

🚨 Red Flags Detected:
   🚩 No company logo (+5%)
   🚩 Urgency language: asap (+8%)
   🚩 Very short description (107 chars) (+6%)
   🚩 No salary information (+3%)

🤖 Model Scores:
   - Naive Bayes: 0.3390
   - LSTM: 0.9670
   - Agreement: False

🎯 Strategy: Rule-boosted ensemble (base: 81.0% + boost: 22.0% = 95.0%)

🚨 Top Fraud Indicators:
   1. 'rep': weight=1.670
   2. 'explain': weight=1.237
   3. 'needed': weight=1.513
   4. 'customer service': weight=0.910
   5. 'phone': weight=1.440

⚠️ Risky Sentences:
   [34%] customer service rep US, CA, sacremento customer service reps needed asap  will ...
       🚩 Urgency indicator

💡 Recommendation: DO NOT APPLY - Strong fraud indicators detected


Test Job (Real, Not Fraud)



In [22]:
# Get a real job
real_jobs = df[df['fraudulent'] == 0]
real_sample = real_jobs.sample(1, random_state=42).iloc[0]

print("\n" + "="*80)
print("TEST 2: LEGITIMATE JOB FROM DATASET")
print("="*80 + "\n")

print(f"Job ID: {real_sample['job_id']}")
print(f"Title: {safe_display_field(real_sample, 'title')}")
print(f"Company: {safe_display_field(real_sample, 'company_profile')}")
print(f"Actual Label: LEGITIMATE\n")

real_text_real = combine_text_fields(real_sample)

real_result = ensemble.predict_with_explanation(
    text=real_text_real,
    telecommuting=int(real_sample.get('telecommuting', 0)),
    has_company_logo=int(real_sample.get('has_company_logo', 0)),
    has_questions=int(real_sample.get('has_questions', 0)),
    ensemble_strategy='rule_boosted'
)

print(f"🔍 FRAUD PROBABILITY: {real_result['fraud_probability_pct']}")
print(f"{real_result['risk_icon']} RISK LEVEL: {real_result['risk_level']}")
print(f"📊 PREDICTED: {real_result['ensemble_decision']}")
print(f"✅ CONFIDENCE: {real_result['confidence']}")
print(f"🎯 CORRECT: {real_result['ensemble_decision'] == 'REAL'}\n")

print(f"🤖 Model Scores:")
print(f"   - Naive Bayes: {real_result['individual_predictions']['naive_bayes_score']}")
print(f"   - LSTM: {real_result['individual_predictions']['lstm_score']}")
print(f"   - Agreement: {real_result['individual_predictions']['agreement']}\n")

print(f"💡 Recommendation: {real_result['recommendation']}")



TEST 2: LEGITIMATE JOB FROM DATASET

Job ID: 5231
Title: SEM Coordinator
Company: #URL_c379aa631173ed5b7c345ab3f500a9a053e509138ca70e52c1088e5a784dc8d7# is a modern online travel age...
Actual Label: LEGITIMATE

🔍 FRAUD PROBABILITY: 3.2%
🟢 RISK LEVEL: LOW RISK
📊 PREDICTED: REAL
✅ CONFIDENCE: HIGH
🎯 CORRECT: True

🤖 Model Scores:
   - Naive Bayes: 0.0000
   - LSTM: 0.0030
   - Agreement: True

💡 Recommendation: APPEARS SAFE - Standard application precautions recommended


Test Jobs (Real, Multiple)

In [24]:
fraud_samples = fraud_jobs.sample(5, random_state=123)
real_samples = real_jobs.sample(5, random_state=456)

results_list = []

print("Testing 10 job postings with rule-boosted ensemble...\n")

# Process fraud jobs
for idx, row in fraud_samples.iterrows():
    text = combine_text_fields(row)
    result = ensemble.predict_with_explanation(
        text=text,
        telecommuting=int(row.get('telecommuting', 0)),
        has_company_logo=int(row.get('has_company_logo', 0)),
        has_questions=int(row.get('has_questions', 0)),
        ensemble_strategy='rule_boosted'
    )
    results_list.append({
        'Job ID': row['job_id'],
        'True Label': 'FRAUD',
        'Risk Level': result['risk_level'],
        'Predicted': result['ensemble_decision'],
        'Fraud Prob': f"{result['fraud_probability']*100:.1f}%",
        'Correct': result['ensemble_decision'] == 'FRAUD',
        'NB Score': result['individual_predictions']['naive_bayes_score'],
        'LSTM Score': result['individual_predictions']['lstm_score']
    })

# Process real jobs
for idx, row in real_samples.iterrows():
    text = combine_text_fields(row)
    result = ensemble.predict_with_explanation(
        text=text,
        telecommuting=int(row.get('telecommuting', 0)),
        has_company_logo=int(row.get('has_company_logo', 0)),
        has_questions=int(row.get('has_questions', 0)),
        ensemble_strategy='rule_boosted'
    )
    results_list.append({
        'Job ID': row['job_id'],
        'True Label': 'REAL',
        'Risk Level': result['risk_level'],
        'Predicted': result['ensemble_decision'],
        'Fraud Prob': f"{result['fraud_probability']*100:.1f}%",
        'Correct': result['ensemble_decision'] == 'REAL',
        'NB Score': result['individual_predictions']['naive_bayes_score'],
        'LSTM Score': result['individual_predictions']['lstm_score']
    })

# Display results
results_df = pd.DataFrame(results_list)
print(results_df.to_string(index=False))

accuracy = results_df['Correct'].sum() / len(results_df)
print(f"\n📊 Accuracy on sample: {accuracy*100:.1f}%")
print(f"✅ Correct predictions: {results_df['Correct'].sum()}/{len(results_df)}")

# Breakdown by risk level
print(f"\n🎯 Risk Level Breakdown:")
for risk in ['HIGH RISK', 'MEDIUM RISK', 'LOW RISK']:
    count = (results_df['Risk Level'] == risk).sum()
    print(f"   {risk}: {count} jobs")

Testing 10 job postings with rule-boosted ensemble...

 Job ID True Label  Risk Level Predicted Fraud Prob  Correct NB Score LSTM Score
   5585      FRAUD MEDIUM RISK     FRAUD      52.1%     True   0.0009     0.3872
  17564      FRAUD   HIGH RISK     FRAUD      95.0%     True   0.9617     0.9552
   6501      FRAUD MEDIUM RISK      REAL      44.7%    False   0.0068     0.3673
   2010      FRAUD MEDIUM RISK     FRAUD      54.1%     True   0.0019     0.4146
   4681      FRAUD   HIGH RISK     FRAUD      94.3%     True   0.3340     0.9721
  17189       REAL    LOW RISK      REAL      13.3%     True   0.0006     0.0041
  13708       REAL    LOW RISK      REAL       8.1%     True   0.0000     0.0019
  16171       REAL    LOW RISK      REAL      15.5%     True   0.0582     0.0270
   9425       REAL    LOW RISK      REAL      17.2%     True   0.0000     0.0021
   2691       REAL MEDIUM RISK      REAL      25.0%     True   0.0003     0.0003

📊 Accuracy on sample: 90.0%
✅ Correct predictions: 9/

In [31]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# ============================================================================
# OPTIMIZED BATCH ENSEMBLE CLASS
# ============================================================================

class FastInterpretableEnsemble(InterpretableEnsemble):
    """Optimized batch processing version - 20-50x faster"""

    def predict_batch(self, texts, telecommuting_arr, logo_arr, questions_arr):
        """
        Batch prediction for multiple jobs

        Args:
            texts: List/array of job texts
            telecommuting_arr: Array of telecommuting flags
            logo_arr: Array of has_company_logo flags
            questions_arr: Array of has_questions flags

        Returns:
            DataFrame with predictions
        """
        n_samples = len(texts)
        print(f"Batch processing {n_samples} samples...")

        # ===== BATCH PREPROCESS =====
        print("  1/5 Preprocessing text...")
        texts_processed = [self._preprocess_text(text) for text in tqdm(texts, desc="    Preprocessing", leave=False)]

        # ===== BATCH NB PREDICTION =====
        print("  2/5 NB prediction...")
        nb_text_features = self.vectorizer.transform(texts_processed)

        # Create numeric features array
        numeric_features_arr = np.column_stack([
            telecommuting_arr,
            logo_arr,
            questions_arr,
            np.full(n_samples, 0.05),  # location_fraud_ratio
            [len(text) for text in texts]  # character_count
        ])

        numeric_features_sparse = csr_matrix(numeric_features_arr)
        nb_combined = hstack([nb_text_features, numeric_features_sparse])

        # Batch predict with NB
        nb_probs = self.nb.predict_proba(nb_combined)[:, 1]

        # ===== BATCH LSTM PREDICTION =====
        print("  3/5 LSTM prediction...")
        lstm_sequences = self.tokenizer.texts_to_sequences(texts_processed)
        lstm_padded = pad_sequences(lstm_sequences, maxlen=200)

        # Batch predict with LSTM (128 samples at a time)
        lstm_probs = self.lstm.predict(lstm_padded, batch_size=128, verbose=0).flatten()

        # ===== BATCH ENSEMBLE =====
        print("  4/5 Computing ensemble scores...")
        base_probs = 0.25 * nb_probs + 0.75 * lstm_probs

        # ===== BATCH RULE-BOOSTING =====
        print("  5/5 Applying red flags...")
        boosts = np.zeros(n_samples)

        # Vectorized red flags
        boosts += np.where(logo_arr == 0, 0.05, 0)
        boosts += np.where(questions_arr == 0, 0.05, 0)
        boosts += np.where([len(t) < 300 for t in texts], 0.06, 0)

        # Keyword checks (optimized with set operations)
        urgency_kw = {'urgent', 'asap', 'immediately'}
        payment_kw = {'payment', 'fee', 'upfront'}

        for i, text_lower in enumerate([t.lower() for t in texts]):
            words = set(text_lower.split())
            if words & urgency_kw:  # Intersection check (faster)
                boosts[i] += 0.08
            if words & payment_kw:
                boosts[i] += 0.12

        fraud_probs = np.minimum(base_probs + boosts, 0.95)

        # ===== CLASSIFICATION =====
        predictions = np.where(fraud_probs > 0.5, 'FRAUD', 'REAL')
        risk_levels = np.where(fraud_probs > 0.6, 'HIGH RISK',
                              np.where(fraud_probs > 0.2, 'MEDIUM RISK', 'LOW RISK'))

        print("✓ Batch processing complete!\n")

        # Return DataFrame
        return pd.DataFrame({
            'fraud_prob': fraud_probs,
            'predicted': predictions,
            'risk_level': risk_levels,
            'nb_score': nb_probs,
            'lstm_score': lstm_probs
        })

# ============================================================================
# FAST TEST SET EVALUATION
# ============================================================================

print("="*80)
print("FAST BATCH EVALUATION - RULE-BOOSTED ENSEMBLE")
print("="*80 + "\n")

# Prepare the data
print("Preparing data...")
df['text_combined'] = df.apply(combine_text_fields, axis=1)

# TF-IDF Vectorization
X_text = vectorizer.transform(df['text_combined'])

# Numeric features
numeric_features_all_df = df[['telecommuting', 'has_company_logo', 'has_questions']].fillna(0).astype(int)
numeric_features_all_df['location_fraud_ratio'] = 0.05
numeric_features_all_df['character_count'] = df['text_combined'].apply(len)

X_numeric_sparse = csr_matrix(numeric_features_all_df.values)
y = df['fraudulent'].values

# Split dataset
X_combined_all = hstack([X_text, X_numeric_sparse])

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X_combined_all, y, df,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Test set: {len(df_test)} samples")
print(f"  - Fraud cases: {y_test.sum()}")
print(f"  - Real cases: {(~y_test.astype(bool)).sum()}\n")

# ===== FAST BATCH PREDICTION =====

# Initialize fast ensemble
fast_ensemble = FastInterpretableEnsemble(nb_model, lstm_model, vectorizer, tokenizer)

# Extract test data
test_texts = df_test['text_combined'].tolist()
test_telecommuting = df_test['telecommuting'].fillna(0).astype(int).values
test_logo = df_test['has_company_logo'].fillna(0).astype(int).values
test_questions = df_test['has_questions'].fillna(0).astype(int).values
test_labels = df_test['fraudulent'].values
test_job_ids = df_test['job_id'].values

# Batch predict (20-50x faster!)
results_df = fast_ensemble.predict_batch(
    texts=test_texts,
    telecommuting_arr=test_telecommuting,
    logo_arr=test_logo,
    questions_arr=test_questions
)

# Add ground truth and job IDs
results_df['job_id'] = test_job_ids
results_df['true_label'] = np.where(test_labels == 1, 'FRAUD', 'REAL')
results_df['correct'] = results_df['predicted'] == results_df['true_label']

# ============================================================================
# PERFORMANCE METRICS (same as before)
# ============================================================================

print("="*80)
print("PERFORMANCE SUMMARY")
print("="*80 + "\n")

# Overall accuracy
accuracy = results_df['correct'].sum() / len(results_df)
print(f"📊 Binary Accuracy: {accuracy*100:.2f}%")
print(f"   Correct: {results_df['correct'].sum()}/{len(results_df)}\n")

# Breakdown by true label
fraud_df = results_df[results_df['true_label'] == 'FRAUD']
real_df = results_df[results_df['true_label'] == 'REAL']

print(f"🚨 Fraud Detection Performance:")
print(f"   Total fraud cases: {len(fraud_df)}")
print(f"   Correctly identified: {fraud_df['correct'].sum()}")
print(f"   Accuracy on fraud: {fraud_df['correct'].mean()*100:.2f}%\n")

print(f"✅ Legitimate Job Performance:")
print(f"   Total real cases: {len(real_df)}")
print(f"   Correctly identified: {real_df['correct'].sum()}")
print(f"   Accuracy on real: {real_df['correct'].mean()*100:.2f}%\n")

# Risk level breakdown
print(f"🎯 Risk Level Distribution:")
for risk in ['HIGH RISK', 'MEDIUM RISK', 'LOW RISK']:
    count = (results_df['risk_level'] == risk).sum()
    fraud_count = ((results_df['risk_level'] == risk) &
                   (results_df['true_label'] == 'FRAUD')).sum()
    real_count = ((results_df['risk_level'] == risk) &
                  (results_df['true_label'] == 'REAL')).sum()
    print(f"   {risk}: {count} total (Fraud: {fraud_count}, Real: {real_count})")

# Fraud detection by risk level
print(f"\n🔍 Fraud Cases by Risk Level:")
fraud_high = ((results_df['true_label'] == 'FRAUD') &
              (results_df['risk_level'] == 'HIGH RISK')).sum()
fraud_medium = ((results_df['true_label'] == 'FRAUD') &
                (results_df['risk_level'] == 'MEDIUM RISK')).sum()
fraud_low = ((results_df['true_label'] == 'FRAUD') &
             (results_df['risk_level'] == 'LOW RISK')).sum()

print(f"   🔴 HIGH RISK: {fraud_high}/{len(fraud_df)} ({fraud_high/len(fraud_df)*100:.1f}%)")
print(f"   🟡 MEDIUM RISK: {fraud_medium}/{len(fraud_df)} ({fraud_medium/len(fraud_df)*100:.1f}%)")
print(f"   🟢 LOW RISK (missed): {fraud_low}/{len(fraud_df)} ({fraud_low/len(fraud_df)*100:.1f}%)\n")

fraud_caught = fraud_high + fraud_medium
print(f"   📈 Total Fraud Flagged (HIGH + MEDIUM): {fraud_caught}/{len(fraud_df)} ({fraud_caught/len(fraud_df)*100:.1f}%)")

# False positive analysis
print(f"\n⚠️ False Positive Analysis:")
fp_high = ((results_df['true_label'] == 'REAL') &
           (results_df['risk_level'] == 'HIGH RISK')).sum()
fp_medium = ((results_df['true_label'] == 'REAL') &
             (results_df['risk_level'] == 'MEDIUM RISK')).sum()

print(f"   Real jobs flagged as HIGH RISK: {fp_high}/{len(real_df)} ({fp_high/len(real_df)*100:.1f}%)")
print(f"   Real jobs flagged as MEDIUM RISK: {fp_medium}/{len(real_df)} ({fp_medium/len(real_df)*100:.1f}%)")

# Precision, Recall, F1
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true_binary = (results_df['true_label'] == 'FRAUD').astype(int)
y_pred_binary = (results_df['predicted'] == 'FRAUD').astype(int)

precision = precision_score(y_true_binary, y_pred_binary)
recall = recall_score(y_true_binary, y_pred_binary)
f1 = f1_score(y_true_binary, y_pred_binary)

print(f"\n📐 Binary Classification Metrics:")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}\n")

# Confusion matrix
cm = confusion_matrix(y_true_binary, y_pred_binary)
tn, fp, fn, tp = cm.ravel()

print(f"🎯 Confusion Matrix:")
print(f"   True Negatives (correct real): {tn}")
print(f"   False Positives (real → fraud): {fp}")
print(f"   False Negatives (fraud → real): {fn}")
print(f"   True Positives (correct fraud): {tp}")

# Model comparison
print(f"\n🤖 Model Score Statistics:")
print(f"   NB Mean (Fraud): {fraud_df['nb_score'].mean():.4f}")
print(f"   LSTM Mean (Fraud): {fraud_df['lstm_score'].mean():.4f}")
print(f"   NB Mean (Real): {real_df['nb_score'].mean():.4f}")
print(f"   LSTM Mean (Real): {real_df['lstm_score'].mean():.4f}")


FAST BATCH EVALUATION - RULE-BOOSTED ENSEMBLE

Preparing data...
Test set: 5364 samples
  - Fraud cases: 260
  - Real cases: 5104

✓ InterpretableEnsemble initialized
Batch processing 5364 samples...
  1/5 Preprocessing text...


  2/5 NB prediction...
  3/5 LSTM prediction...
  4/5 Computing ensemble scores...
  5/5 Applying red flags...
✓ Batch processing complete!

PERFORMANCE SUMMARY

📊 Binary Accuracy: 97.52%
   Correct: 5231/5364

🚨 Fraud Detection Performance:
   Total fraud cases: 260
   Correctly identified: 166
   Accuracy on fraud: 63.85%

✅ Legitimate Job Performance:
   Total real cases: 5104
   Correctly identified: 5065
   Accuracy on real: 99.24%

🎯 Risk Level Distribution:
   HIGH RISK: 171 total (Fraud: 154, Real: 17)
   MEDIUM RISK: 275 total (Fraud: 46, Real: 229)
   LOW RISK: 4918 total (Fraud: 60, Real: 4858)

🔍 Fraud Cases by Risk Level:
   🔴 HIGH RISK: 154/260 (59.2%)
   🟡 MEDIUM RISK: 46/260 (17.7%)
   🟢 LOW RISK (missed): 60/260 (23.1%)

   📈 Total Fraud Flagged (HIGH + MEDIUM): 200/260 (76.9%)

⚠️ False Positive Analysis:
   Real jobs flagged as HIGH RISK: 17/5104 (0.3%)
   Real jobs flagged as MEDIUM RISK: 229/5104 (4.5%)

📐 Binary Classification Metrics:
   Precision: 0.8098
   Reca